# Predict Shapes From A Saved Architecture Model

This notebook loads a saved `run_architectures.py` experiment directory containing `config.json` and `model.eqx`, runs the trained model on a dataset `.npz`, and saves the predicted trajectory to a new `.npz` file.

In [1]:
import json
import os
import sys
from pathlib import Path

os.environ.setdefault("XLA_PYTHON_CLIENT_PREALLOCATE", "false")

import equinox as eqx
import jax
import jax.numpy as jnp
import numpy as np

jax.config.update("jax_enable_x64", True)

cwd = Path.cwd()
if (cwd / "run_architectures.py").is_file():
    SLINKY_2D = cwd
elif (cwd / "examples/slinky/slinky_2D/run_architectures.py").is_file():
    SLINKY_2D = cwd / "examples/slinky/slinky_2D"
else:
    raise RuntimeError("Run this notebook from the repo root or examples/slinky/slinky_2D.")

sys.path.insert(0, str(SLINKY_2D))

from run_architectures import (
    _properties_from_payload,
    _resolve_saved_data_path,
    _sweep_config_from_payload,
    build_architecture_registry,
    make_model_params,
)
from util import Dataset, get_slinky, predict

ROOT = SLINKY_2D.resolve()
ROOT

PosixPath('/Users/radha/GitRepos/dismech-jax/examples/slinky/slinky_2D')

## User Inputs

Set `RUN_DIR` to the saved experiment directory. It should contain `config.json` and `model.eqx`.

Set `DATA_FILE` to your dataset `.npz`. If you leave it as `None`, the notebook uses the `valid_file` path saved in `config.json`.

In [2]:
# Directory made by run_architectures.py for a single trained architecture.
# RUN_DIR = Path("arch_sweep_outputs_n11_tape_all_architectures_train_fail_on_nonconvergence_True/" \
# "brazier_chol_stiffness_icnn__hid_10__inp_raw__stretchNN_0__bendNN_0__act_tanh__corr_0.01__zr1__seed_42__mdl_0.05__it_10__hreg_1e-06__hprobe_1__hseed_0__wd_1e-05__mode_anisotropic")

# RUN_DIR = Path("arch_sweep_outputs_n11_tape_all_architectures_train_fail_on_nonconvergence_True/" \
# "diag_energy_baseline__hid_10__inp_raw__stretchNN_0__bendNN_0__act_tanh__corr_0.01__zr1__seed_42__mdl_0.05__it_10__hreg_1e-06__hprobe_1__hseed_0__wd_1e-05")

RUN_DIR = Path("N9 strip architecture sweep /arch_sweep_outputs_n9_strip_all_architectures_failOnNonconvergence_True/diag_energy_baseline__hid_10__inp_raw__stretchNN_0__bendNN_0__act_tanh__corr_0.01__zr1__seed_42__mdl_0.01__it_20__hreg_1e-06__hprobe_1__hseed_0")


# Dataset with qs, xb, idx_b, lambdas, valid. Use None to load the saved valid_file.
# DATA_FILE = "../experiment_data/tape_data/11_noded/n11_tape_train_dataset.npz"
# DATA_FILE = None
DATA_FILE = "../experiment_data/n9_strip_train_dataset.npz"

# Used only for naming the default output file.
SPLIT_NAME = "train"
# SPLIT_NAME = "valid"

# Use None to save next to RUN_DIR as <split>_predicted_shapes.npz.
OUTPUT_FILE = None

# If True, save only qs_pred. If False, also save dataset arrays and metadata.
SAVE_MINIMAL = True

# Optional overrides for the continuation/Newton solve.
# Example: SOLVER_OVERRIDES = {"max_dlambda": 1e-2, "iters": 10}
SOLVER_OVERRIDES = {}

RUN_DIR = RUN_DIR.expanduser()
if not RUN_DIR.is_absolute():
    RUN_DIR = ROOT / RUN_DIR
RUN_DIR = RUN_DIR.resolve()
if DATA_FILE is not None:
    DATA_FILE = Path(DATA_FILE).expanduser()
    if not DATA_FILE.is_absolute():
        DATA_FILE = ROOT / DATA_FILE
    DATA_FILE = DATA_FILE.resolve()
if OUTPUT_FILE is not None:
    OUTPUT_FILE = Path(OUTPUT_FILE).expanduser()
    if not OUTPUT_FILE.is_absolute():
        OUTPUT_FILE = ROOT / OUTPUT_FILE
    OUTPUT_FILE = OUTPUT_FILE.resolve()

RUN_DIR

PosixPath('/Users/radha/GitRepos/dismech-jax/examples/slinky/slinky_2D/N9 strip architecture sweep /arch_sweep_outputs_n9_strip_all_architectures_failOnNonconvergence_True/diag_energy_baseline__hid_10__inp_raw__stretchNN_0__bendNN_0__act_tanh__corr_0.01__zr1__seed_42__mdl_0.01__it_20__hreg_1e-06__hprobe_1__hseed_0')

In [3]:
config_path = RUN_DIR / "config.json"
model_path = RUN_DIR / "model.eqx"

if not config_path.is_file():
    raise FileNotFoundError(f"Missing config.json: {config_path}")
if not model_path.is_file():
    raise FileNotFoundError(f"Missing model.eqx: {model_path}")

with config_path.open() as f:
    payload = json.load(f)

cfg = _sweep_config_from_payload(payload)
registry = build_architecture_registry()
arch_name = payload["arch_name"]
spec = registry[arch_name]

properties = _properties_from_payload(payload)
for attr in ("start", "end"):
    value = getattr(properties, attr, None)
    if value is not None:
        setattr(properties, attr, jnp.asarray(value))

params = make_model_params(cfg, spec)
model = eqx.tree_deserialise_leaves(str(model_path), spec.model_cls(params))

if DATA_FILE is None:
    DATA_FILE = Path(_resolve_saved_data_path(payload, "valid_file", str(RUN_DIR))).resolve()

data = Dataset.load(str(DATA_FILE), force_key=payload.get("force_key"))
base, aux = get_slinky(properties)

print(f"Loaded architecture : {arch_name}")
print(f"Model              : {model_path}")
print(f"Dataset            : {DATA_FILE}")
print(f"qs shape           : {data.qs.shape}")
print(f"xb shape           : {data.xb.shape}")
print(f"idx_b shape        : {data.idx_b.shape}")
print(f"lambdas shape      : {data.lambdas.shape}")
print(f"valid shape        : {data.valid.shape}")

Loaded architecture : diag_energy_baseline
Model              : /Users/radha/GitRepos/dismech-jax/examples/slinky/slinky_2D/N9 strip architecture sweep /arch_sweep_outputs_n9_strip_all_architectures_failOnNonconvergence_True/diag_energy_baseline__hid_10__inp_raw__stretchNN_0__bendNN_0__act_tanh__corr_0.01__zr1__seed_42__mdl_0.01__it_20__hreg_1e-06__hprobe_1__hseed_0/model.eqx
Dataset            : /Users/radha/GitRepos/dismech-jax/examples/slinky/experiment_data/n9_strip_train_dataset.npz
qs shape           : (3, 28, 35)
xb shape           : (3, 28, 20)
idx_b shape        : (20,)
lambdas shape      : (3, 28)
valid shape        : (3, 28)


In [4]:
solver_kwargs = dict(
    max_dlambda=cfg.max_dlambda,
    iters=cfg.iters,
    ls_steps=cfg.ls_steps,
    abs_tol=cfg.abs_tol,
    rel_tol=cfg.rel_tol,
    fail_on_nonconvergence=cfg.prediction_fail_on_nonconvergence,
    early_stop=cfg.early_stop,
)
solver_kwargs.update(SOLVER_OVERRIDES)

pred = predict(
    model,
    base,
    aux,
    data.idx_b,
    data.xb,
    data.lambdas,
    **solver_kwargs,
)

pred_np = np.asarray(pred, dtype=float)
print(f"pred shape         : {pred_np.shape}")

pred shape         : (3, 28, 35)


In [5]:
if OUTPUT_FILE is None:
    OUTPUT_FILE = RUN_DIR / f"{SPLIT_NAME}_predicted_shapes.npz"

OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)

if SAVE_MINIMAL:
    np.savez(OUTPUT_FILE, qs_pred=pred_np)
else:
    np.savez(
        OUTPUT_FILE,
        qs_pred=pred_np,
        qs=np.asarray(data.qs, dtype=float),
        xb=np.asarray(data.xb, dtype=float),
        idx_b=np.asarray(data.idx_b),
        lambdas=np.asarray(data.lambdas, dtype=float),
        valid=np.asarray(data.valid, dtype=bool),
        dataset_file=str(DATA_FILE),
        run_dir=str(RUN_DIR),
        arch_name=arch_name,
        model_cls=spec.model_cls.__name__,
        which_case=spec.which_case,
        solver_kwargs=json.dumps(solver_kwargs),
    )

print(f"Saved predictions  : {OUTPUT_FILE}")

Saved predictions  : /Users/radha/GitRepos/dismech-jax/examples/slinky/slinky_2D/N9 strip architecture sweep /arch_sweep_outputs_n9_strip_all_architectures_failOnNonconvergence_True/diag_energy_baseline__hid_10__inp_raw__stretchNN_0__bendNN_0__act_tanh__corr_0.01__zr1__seed_42__mdl_0.01__it_20__hreg_1e-06__hprobe_1__hseed_0/train_predicted_shapes.npz


In [6]:
saved = np.load(OUTPUT_FILE)
print(saved.files)
print(saved["qs_pred"].shape)

['qs_pred']
(3, 28, 35)
